In [33]:
# Install client dependencies if running in a clean runtime environment
!pip install --quiet ollama pydantic pillow numpy matplotlib

In [34]:
import os
import json
import base64
import ollama
from io import BytesIO
from PIL import Image, ImageDraw
from pydantic import BaseModel, Field, ValidationError
from typing import List, Optional

print("Ollama Client Version Library Verified.")

Ollama Client Version Library Verified.


In [35]:
# 1. Install the Ollama CLI wrapper if running in a cloud notebook (e.g., Google Colab)
!sudo apt-get install zstd

!curl -fsSL https://ollama.com/install.sh | sh

# 2. Start the Ollama background daemon service
import subprocess
import time

print("Starting Ollama background server...")
# Launch the server without blocking the notebook execution
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Give the server a few seconds to fully initialize
time.sleep(5)
print("Ollama server should be live.")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Starting Ollama background server...
Ollama server should be live.


In [36]:
# Pull the specific multimodal model required by your script
!ollama pull qwen2.5vl:7b

In [37]:
# Check local model availability
!ollama list

NAME            ID              SIZE      MODIFIED               
qwen2.5vl:7b    5ced39dfa4ba    6.0 GB    Less than a second ago    


In [38]:
def create_medical_invoice() -> str:
    """Generates a mock medical claim document for model analysis."""
    img = Image.new('RGB', (700, 450), color=(255, 255, 255))
    draw = ImageDraw.Draw(img)

    # Structural Content
    draw.text((40, 30), "METRO HEALTH CLINIC", fill=(20, 20, 20))
    draw.text((40, 70), "Attending Physician: Dr. Robert Chen, MD", fill=(50, 50, 50))
    draw.text((40, 100), "Patient Name: Eleanor Vance", fill=(50, 50, 50))
    draw.text((40, 130), "Date of Service: May 12, 2026", fill=(50, 50, 50))

    draw.line([(40, 160), (660, 160)], fill=(200, 200, 200), width=2)

    draw.text((40, 180), "DIAGNOSIS: Acute Strep Pharyngitis", fill=(30, 30, 30))
    draw.text((40, 210), "Rx Prescribed: Amoxicillin 500mg (Capsules)", fill=(30, 30, 30))

    draw.line([(40, 250), (660, 250)], fill=(200, 200, 200), width=1)

    draw.text((40, 280), "Itemized Ledger Billing Summary:", fill=(50, 50, 50))
    draw.text((60, 310), "- Consultative Exam Fee:               $120.00", fill=(70, 70, 70))
    draw.text((60, 340), "- Rapid Strep Culture Assay:            $45.00", fill=(70, 70, 70))
    draw.text((40, 380), "TOTAL AMOUNT DUE AND PAID:            $165.00", fill=(10, 10, 10))

    img_path = "medical_invoice.png"
    img.save(img_path)
    return img_path

invoice_file = create_medical_invoice()
print(f"Mock medical receipt cached to disk at: {invoice_file}")

Mock medical receipt cached to disk at: medical_invoice.png


In [39]:
class InsuranceExtractionSchema(BaseModel):
    """The target JSON structure for claims validation."""
    patient_name: str = Field(..., description="Full legal name of the patient.")
    doctor_name: str = Field(..., description="Name of the attending medical practitioner.")
    hospital_name: str = Field(..., description="The name of the clinic or healthcare facility.")
    total_amount: float = Field(..., description="The numerical balance due or total paid amount.")
    diagnosis: str = Field(..., description="The clinical assessment summary text.")
    medicines: List[str] = Field(..., description="Array of all specified drugs or pharmaceutical agents.")
    service_date: str = Field(..., description="The date the healthcare services were rendered (YYYY-MM-DD or raw string format).")

In [42]:
def extract_structured_claim(image_path: str, raw_ocr_text: Optional[str] = None) -> Optional[InsuranceExtractionSchema]:
    """
    Streams an image document alongside grounding text tokens into
    local Qwen2.5-VL context, returning a type-safe schema payload.
    """
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Source file not located: {image_path}")

    # Standardize image array data into base64 bytecode payloads for Ollama multi-modal API ingestion
    with open(image_path, "rb") as image_file:
        b64_image_bytes = base64.b64encode(image_file.read()).decode('utf-8')

    # Inject OCR grounding strings to minimize hallucination limits if available
    grounding_clause = f"\nUse this verified OCR text to double-check spelling and figures:\n{raw_ocr_text}" if raw_ocr_text else ""

    system_prompt = (
        "You are an enterprise insurance parsing agent. Your job is to extract data from incoming document scans.\n"
        "You must return data fitting the following JSON specification structure strictly:\n"
        "{\n"
        '  "patient_name": "string",\n'
        '  "doctor_name": "string",\n'
        '  "hospital_name": "string",\n'
        '  "total_amount": 0.0,\n'
        '  "diagnosis": "string",\n'
        '  "medicines": ["string"],\n'
        '  "service_date": "string"\n'
        "}\n"
        "CRITICAL: Output ONLY valid, parsable JSON text code wrappers. Do not include markdown commentary, introductions, or closing remarks."
    )

    user_prompt = f"Extract the target insurance details from this document scan. {grounding_clause}"

    try:
        # Invoke local Ollama chat controller service
        response = ollama.chat(
            model='qwen2.5vl:7b',
            messages=[
                {
                    'role': 'system',
                    'content': system_prompt
                },
                {
                    'role': 'user',
                    'content': user_prompt,
                    'images': [b64_image_bytes]
                }
            ],
            options={
                'temperature': 0.0 # Set to absolute 0 to minimize creativity and maximize deterministic extractions
            }
        )

        raw_output = response['message']['content'].strip()

        # Strip away any markdown formatting if present
        if raw_output.startswith("```json"):
            raw_output = raw_output[7:]
        if raw_output.endswith("```"):
            raw_output = raw_output[:-3]
        raw_output = raw_output.strip()

        # Parse the output into our Pydantic schema
        json_dict = json.loads(raw_output)
        validated_payload = InsuranceExtractionSchema(**json_dict)
        return validated_payload

    except json.JSONDecodeError as je:
        print(f"❌ Structural Failure: Output was not valid JSON text context -> {je}")
        print(f"Raw Output: {raw_output}")
    # CHANGED: replaced 'excitement' with 'as e' (or any valid variable name)
    except ValidationError as e:
        print(f"❌ Schema Type Binding Invalidation Exception -> {e}")
    except Exception as e:
        print(f"⚠️ Runtime processing block fault -> {str(e)}")

    return None

# Execute extraction without explicit OCR text grounding first
print("Running local VLM visual extraction step...")
extracted_data = extract_structured_claim(invoice_file)

if extracted_data:
    print("\n🎉 Extraction Successful! Verified Output Object:")
    print(extracted_data.model_dump_json(indent=2))

Running local VLM visual extraction step...

🎉 Extraction Successful! Verified Output Object:
{
  "patient_name": "Eleanor Vance",
  "doctor_name": "Dr. Robert Chen, MD",
  "hospital_name": "METRO HEALTH CLINIC",
  "total_amount": 165.0,
  "diagnosis": "Acute Strep Pharyngitis",
  "medicines": [
    "Amoxicillin 500mg (Capsules)"
  ],
  "service_date": "May 12, 2026"
}
